# Day 5: Feature Selection & Target Engineering

## Objective
Prepare production-grade features before model training.

## Steps
1. Target analysis (price distribution, outliers)
2. Binary target creation (`high_price_period`)
3. Null & data quality check
4. Correlation analysis
5. Feature importance (quick RF)
6. Train/test split (time-based)
7. Class imbalance check
8. Save final ML-ready datasets

In [0]:
%run ./00_config

In [0]:
df = spark.table(T_GOLD_MODEL_SCORING)
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
print("\nSchema:")
for f in df.schema.fields:
    print(f"  {f.name:40s} {f.dataType.simpleString()}")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

pdf = df.select("price_eur_mwh").toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(pdf["price_eur_mwh"], bins=80, edgecolor="black", alpha=0.7)
axes[0].set_title("Price Distribution")
axes[0].set_xlabel("EUR/MWh")
axes[0].axvline(pdf["price_eur_mwh"].quantile(0.75), color="red", linestyle="--", label="75th pctl")
axes[0].legend()

# Box plot
axes[1].boxplot(pdf["price_eur_mwh"], vert=True)
axes[1].set_title("Price Box Plot (outliers)")
axes[1].set_ylabel("EUR/MWh")

# Time series
ts = df.select("event_time_utc", "price_eur_mwh").orderBy("event_time_utc").toPandas()
axes[2].plot(ts["event_time_utc"], ts["price_eur_mwh"], linewidth=0.3, alpha=0.7)
axes[2].set_title("Price Over Time")
axes[2].set_xlabel("Date")

plt.tight_layout()
plt.show()

# Stats
print(f"Min: {pdf['price_eur_mwh'].min():.2f}")
print(f"Max: {pdf['price_eur_mwh'].max():.2f}")
print(f"Mean: {pdf['price_eur_mwh'].mean():.2f}")
print(f"Median: {pdf['price_eur_mwh'].median():.2f}")
print(f"Std: {pdf['price_eur_mwh'].std():.2f}")
print(f"75th pctl: {pdf['price_eur_mwh'].quantile(0.75):.2f}")

In [0]:
# Define threshold at 75th percentile
threshold = df.approxQuantile("price_eur_mwh", [0.75], 0.01)[0]
print(f"High-price threshold (75th pctl): {threshold:.2f} EUR/MWh")

# Create binary label
df = df.withColumn(
    "high_price_period",
    F.when(F.col("price_eur_mwh") > threshold, 1).otherwise(0)
)

# Class distribution
print("\nClass distribution:")
display(df.groupBy("high_price_period").count().withColumn(
    "pct", F.round(F.col("count") / df.count() * 100, 1)
))

In [0]:
print("NULL VALUE CHECK")
print("=" * 60)

total_rows = df.count()
for col_name in df.columns:
    null_count = df.filter(F.col(col_name).isNull()).count()
    pct = null_count / total_rows * 100
    status = "✅" if null_count == 0 else f"⚠️  {null_count:,} ({pct:.1f}%)"
    print(f"  {col_name:40s} {status}")

print(f"\nTotal rows: {total_rows:,}")
print(f"Duplicate timestamps: {total_rows - df.dropDuplicates(['event_time_utc', 'zone']).count():,}")

In [0]:
# Select only numeric feature columns
exclude = {"event_time_utc", "zone", "high_price_period"}
feature_cols = [f.name for f in df.schema.fields 
                if f.dataType.simpleString() in ("double", "int", "bigint", "float")
                and f.name not in exclude]

print(f"Numeric features: {len(feature_cols)}")

# Compute correlation matrix
pdf_features = df.select(feature_cols).toPandas()
corr_matrix = pdf_features.corr()

# Plot
fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(feature_cols, fontsize=8)
plt.colorbar(im)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

# Flag highly correlated pairs (|r| > 0.85)
print("\n⚠️  Highly correlated pairs (|r| > 0.85):")
for i in range(len(feature_cols)):
    for j in range(i+1, len(feature_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.85:
            print(f"  {feature_cols[i]} ↔ {feature_cols[j]}: r={r:.3f}")

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
import pandas as pd

# Exclude target + leakage columns
ml_features = [c for c in feature_cols if c != "price_eur_mwh"]

assembler = VectorAssembler(inputCols=ml_features, outputCol="features", handleInvalid="skip")
df_assembled = assembler.transform(df)

# Quick RF for feature importance (small model)
rf = RandomForestClassifier(
    featuresCol="features", labelCol="high_price_period",
    numTrees=50, maxDepth=8, seed=42
)
rf_model = rf.fit(df_assembled)

# Extract importances
importances = rf_model.featureImportances.toArray()
imp_df = pd.DataFrame({"feature": ml_features, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(imp_df["feature"], imp_df["importance"])
ax.set_xlabel("Importance")
ax.set_title("Feature Importance (RF Classifier)")
plt.tight_layout()
plt.show()

print("\nTop features:")
for _, row in imp_df.sort_values("importance", ascending=False).head(10).iterrows():
    print(f"  {row['feature']:40s} {row['importance']:.4f}")

In [0]:
# Drop features with importance < 1% AND high collinearity
MIN_IMPORTANCE = 0.01

# Force-keep these features regardless of importance
# day_of_week: needed for is_weekend in 05b + day-of-week price analysis
FORCE_KEEP = {"day_of_week"}

selected = imp_df[imp_df["importance"] >= MIN_IMPORTANCE]["feature"].tolist()
dropped = imp_df[imp_df["importance"] < MIN_IMPORTANCE]["feature"].tolist()

# Re-add force-kept features
for feat in FORCE_KEEP:
    if feat in dropped:
        selected.append(feat)
        dropped.remove(feat)
        print(f"  ✅ Force-keeping '{feat}' (needed for downstream engineering)")

# Also drop one from each highly correlated pair (keep the more important one)
# e.g., u10_ms and v10_ms are components of wind_speed_ms
corr_drops = set()
for i in range(len(selected)):
    for j in range(i+1, len(selected)):
        r = pdf_features[selected[i]].corr(pdf_features[selected[j]])
        if abs(r) > 0.90:
            # Drop the less important one (but never a force-kept feature)
            imp_i = imp_df[imp_df["feature"] == selected[i]]["importance"].values[0]
            imp_j = imp_df[imp_df["feature"] == selected[j]]["importance"].values[0]
            drop_col = selected[j] if imp_i >= imp_j else selected[i]
            if drop_col not in FORCE_KEEP:
                corr_drops.add(drop_col)
                print(f"  Dropping '{drop_col}' (correlated with counterpart, r>{0.90})")

final_features = [c for c in selected if c not in corr_drops]

print(f"\n✅ Selected {len(final_features)} features (from {len(ml_features)}):")
for f in final_features:
    print(f"  • {f}")
print(f"\n❌ Dropped {len(dropped) + len(corr_drops)}: {dropped + list(corr_drops)}")

In [0]:
SPLIT_DATE = "2021-01-01"

train = df.filter(F.col("event_time_utc") < F.lit(SPLIT_DATE))
test  = df.filter(F.col("event_time_utc") >= F.lit(SPLIT_DATE))

print(f"Train (2020): {train.count():,} rows")
print(f"Test  (2021): {test.count():,} rows")

# Verify no overlap
overlap = train.select("event_time_utc").intersect(test.select("event_time_utc")).count()
print(f"\nOverlap: {overlap} (should be 0)")

# Class balance in each split
for name, split in [("Train", train), ("Test", test)]:
    total = split.count()
    pos = split.filter(F.col("high_price_period") == 1).count()
    print(f"\n{name} class balance:")
    print(f"  Class 0 (normal):     {total - pos:,} ({(total-pos)/total*100:.1f}%)")
    print(f"  Class 1 (high-price): {pos:,} ({pos/total*100:.1f}%)")

In [0]:
# Calculate class weights for imbalanced training
counts = train.groupBy("high_price_period").count().collect()
count_map = {int(r["high_price_period"]): int(r["count"]) for r in counts}
n0, n1 = count_map[0], count_map[1]
total_train = n0 + n1

w0 = total_train / (2.0 * n0)
w1 = total_train / (2.0 * n1)

print(f"Class 0 count: {n0:,}  weight: {w0:.4f}")
print(f"Class 1 count: {n1:,}  weight: {w1:.4f}")
print(f"\nWeight ratio (w1/w0): {w1/w0:.2f}x")
print("\n✅ Use these weights in model training to handle imbalance")

In [0]:
# Save train/test with selected features + targets
keep_cols = ["event_time_utc", "zone"] + final_features + ["price_eur_mwh", "high_price_period"]

TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.h2_ml_train"
TEST_TABLE  = f"{CATALOG}.{SCHEMA}.h2_ml_test"

train.select(keep_cols).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TRAIN_TABLE)
test.select(keep_cols).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TEST_TABLE)

print(f"\u2705 {TRAIN_TABLE}: {spark.table(TRAIN_TABLE).count():,} rows, {len(keep_cols)} cols")
print(f"\u2705 {TEST_TABLE}:  {spark.table(TEST_TABLE).count():,} rows, {len(keep_cols)} cols")
print(f"\nSelected features: {final_features}")
print(f"Class weights: w0={w0:.4f}, w1={w1:.4f}")
print(f"\n\U0001f389 Day 5 Feature Selection complete! Ready for Day 6 modeling.")